# Quickstart on synthetic phantoms

This notebook walks through the pipeline on synthetic data: generate phantoms in the BraTS folder layout, pre-process them,
create patient-level splits, train a tiny network for a few epochs and look at a prediction.
**The numbers are not BraTS results**; the notebook only shows that every step works.

In [ ]:
import sys, tempfile
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
work = Path(tempfile.mkdtemp(prefix="bratsgan_nb_"))
work

## 1. Phantoms

In [ ]:
from bratsgan.synthetic import make_phantom, write_brats_layout
vols, label, brain = make_phantom(size=48, grade="HGG", seed=1)
z = int(np.argwhere(label > 0)[:, 0].mean())
fig, ax = plt.subplots(1, 5, figsize=(12, 2.8))
for i, name in enumerate(["T1", "T1ce", "T2", "FLAIR"]):
    ax[i].imshow(vols[i, z], cmap="gray"); ax[i].set_title(name)
ax[4].imshow(label[z], cmap="viridis"); ax[4].set_title("labels")
for a in ax: a.axis("off")
plt.show()

## 2. Pre-processing and splits

In [ ]:
from bratsgan.data import find_cases, make_splits, preprocess_case, assert_no_leakage
from bratsgan.utils import write_json
raw = write_brats_layout(work / "raw", n_hgg=12, n_lgg=4, size=40, seed=5)
cases = find_cases(str(raw))
for c in cases:
    preprocess_case(c, work / "cache")
splits = make_splits(cases, seed=1, test_frac=0.25, k=2)
assert_no_leakage(splits)
write_json(work / "splits.json", splits)
{k: (len(v) if k == "test" else [len(f) for f in v]) for k, v in splits.items() if k != "seed"}

## 3. Train a tiny network (adversarial variant A5)

In [ ]:
from bratsgan.config import load_config
from bratsgan.train import run
TINY = dict(base=8, depth=4, max_ch=64, patch=[32, 32, 32], batch_size=2, iters_per_epoch=10, epochs_seg=4, epochs_adv=2,
            val_every=2, num_workers=0, amp=False, adv_ramp_epochs=1, ndf=8)
cfg = load_config(str(ROOT / "configs" / "a5_full.json"),
                  dict(TINY, name="nb", out_dir=str(work / "runs"), cache_dir=str(work / "cache"), splits=str(work / "splits.json")))
out = run(cfg)
out

## 4. Training curve and test metrics

In [ ]:
import pandas as pd
log = pd.read_json(out / "train_log.jsonl", lines=True)
val = log.dropna(subset=["val_dice_mean"])
plt.figure(figsize=(4.5, 3)); plt.plot(val["epoch"], val["val_dice_mean"], marker="o"); plt.xlabel("epoch"); plt.ylabel("validation Dice"); plt.show()
pd.read_csv(out / "test_metrics.csv")[["id", "dice_WT", "dice_TC", "dice_ET", "hd95_WT"]].round(3)

## 5. Segment one test case

In [ ]:
from bratsgan.data import load_case, labels_to_regions
from bratsgan.evaluate import load_for_eval
from bratsgan.infer import predict_regions
from bratsgan.metrics import enforce_hierarchy
import torch
cid = splits["test"][0]
img, seg, _ = load_case(work / "cache", cid, mmap=False)
model = load_for_eval(cfg, str(out / "best.pt"), torch.device("cpu"))
pred, prob, secs = predict_regions(model, np.asarray(img, np.float32), cfg["patch"], 0.5)
pred = enforce_hierarchy(pred); gt = labels_to_regions(np.asarray(seg))
z = int(np.argwhere(gt[0] > 0)[:, 0].mean())
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(img[1, z], cmap="gray"); ax[0].imshow(np.ma.masked_equal(gt[:, z].sum(0), 0), alpha=.6); ax[0].set_title("reference")
ax[1].imshow(img[1, z], cmap="gray"); ax[1].imshow(np.ma.masked_equal(pred[:, z].sum(0), 0), alpha=.6); ax[1].set_title(f"prediction ({secs:.2f} s)")
ax[2].imshow(prob[2, z].astype(float), cmap="magma", vmin=0, vmax=1); ax[2].set_title("P(ET)")
for a in ax: a.axis("off")
plt.show()